# TikTok Data Analysis 
In this notebook I analyze video performace and metrics with the intentions of identifying positive attributes of successful videos in order to impliment those going forward for future data driven success

### 0. Imports

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys, shutil, subprocess, time, random

### 1. Read in the Data

In [18]:
path = "../data/Content.csv"
df = pd.read_csv(path)
print("Num videos:", len(df))
df

Num videos: 15


,Time,Video title,Video link,Post time,Total likes,Total comments,Total shares,Total views
0,February 15,Comment if you can’t speak English 👍,https://www.tiktok.com/@humbletoker/video/7513...,June 7,9940,382,37646,379880
1,February 15,COMMENT IF YOU GOT A #GOODNIGHT !! Did anyone ...,https://www.tiktok.com/@humbletoker/video/7520...,June 25,9275,483,18928,276957
2,February 15,Comment to receive your goodnight ✌️,https://www.tiktok.com/@humbletoker/video/7516...,June 15,8241,1697,24082,272820
3,February 15,NaN,https://www.tiktok.com/@humbletoker/video/7517...,June 19,5882,987,12904,219564
4,February 15,Did anyone win from Super Specific? 🤔 #goodnight,https://www.tiktok.com/@humbletoker/video/7518...,June 22,4797,805,17880,211013
5,February 15,Goodnight to everyone except… Did anyone get a...,https://www.tiktok.com/@humbletoker/video/7518...,June 22,4009,201,18867,203138
6,February 15,🚨Everyone who follows me before 10k gets a spe...,https://www.tiktok.com/@humbletoker/video/7516...,June 16,3909,683,18202,190324
7,February 15,Do they still teach what a mammal is in school...,https://www.tiktok.com/@humbletoker/video/7518...,June 21,3366,255,8377,158644
8,February 15,Comment if you got a goodnight tn 😄,https://www.tiktok.com/@humbletoker/video/7517...,June 17,1431,260,6127,79493
9,February 15,Goodnight to everyone except… Comment if I owe...,https://www.tiktok.com/@humbletoker/video/7519...,June 23,908,69,4835,50998


### 2. Cleaning and Feature Engineering
Vieving the data from I notice a redundant column `Time` so I will drop that column. I also want to observe how commets per view and shares per view correlate with performace (measured in views) so I will create columns for those. I will also add in a rank (by views) columns for quickly identifying how well this video placed. 

In [ ]:
# --- drop redundant column ---
if "Time" in df.columns:
    df = df.drop(columns=["Time"])
df["Video title"] = df["Video title"].fillna("")

# --- primary key: extract video_id from the TikTok URL ---
df["video_id"] = df["Video link"].astype(str).str.extract(r"/video/(\d+)")
df = df.dropna(subset=["video_id"]).copy()
df["video_id"] = df["video_id"].astype(str)
df.head()


,Video title,Video link,Post time,Total likes,Total comments,Total shares,Total views,video_id
0,Comment if you can’t speak English 👍,https://www.tiktok.com/@humbletoker/video/7513...,June 7,9940,382,37646,379880,7513079996349484319
1,COMMENT IF YOU GOT A #GOODNIGHT !! Did anyone ...,https://www.tiktok.com/@humbletoker/video/7520...,June 25,9275,483,18928,276957,7520082772107595063
2,Comment to receive your goodnight ✌️,https://www.tiktok.com/@humbletoker/video/7516...,June 15,8241,1697,24082,272820,7516369352426474798
3,NaN,https://www.tiktok.com/@humbletoker/video/7517...,June 19,5882,987,12904,219564,7517866483989417271
4,Did anyone win from Super Specific? 🤔 #goodnight,https://www.tiktok.com/@humbletoker/video/7518...,June 22,4797,805,17880,211013,7518856355789212983


In [15]:
audio_dir = Path("../data/audio")
audio_dir.mkdir(parents=True, exist_ok=True)

downloaded = skipped = failed = 0
failures = []

for _, row in df.iterrows():
    video_id = str(row["video_id"])
    url = str(row["Video link"])
    out_file = audio_dir / f"{video_id}.m4a"

    # skip if already downloaded
    if out_file.exists() and out_file.stat().st_size > 0:
        skipped += 1
        continue

    cmd = [
        "yt-dlp",
        url,
        "--no-playlist",
        "-x",
        "--audio-format", "m4a",
        "--audio-quality", "0",
        "-o", str(audio_dir / f"{video_id}.%(ext)s"),
        "--force-overwrites",
        "--no-warnings",
    ]

    try:
        print(f"Downloading audio: {video_id}")
        subprocess.run(cmd, check=True)
        downloaded += 1
    except subprocess.CalledProcessError as e:
        failed += 1
        failures.append((video_id, url, str(e)))
        print(f"FAILED: {video_id}")

    # small randomized delay to reduce rate-limits
    time.sleep(random.uniform(0.8, 2.0))

print(f"\nDone. downloaded={downloaded}, skipped={skipped}, failed={failed}")

# show a few failures (if any)
if failures:
    print("\nSample failures:")
    for f in failures[:5]:
        print(f"- {f[0]} | {f[2]}")

[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7513079996349484319
[TikTok] 7513079996349484319: Downloading webpage
[info] 7513079996349484319: Downloading 1 format(s): bytevc1_1080p_306061-1
[download] Destination: ../data/audio/7513079996349484319.mp4
[download] 100% of    2.12MiB in 00:00:00 at 3.51MiB/s   
[ExtractAudio] Destination: ../data/audio/7513079996349484319.m4a
Deleting original file ../data/audio/7513079996349484319.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7520082772107595063
[TikTok] 7520082772107595063: Downloading webpage
[info] 7520082772107595063: Downloading 1 format(s): bytevc1_1080p_266677-1
[download] Destination: ../data/audio/7520082772107595063.mp4
[download] 100% of    2.42MiB in 00:00:00 at 2.64MiB/s   
[ExtractAudio] Destination: ../data/audio/7520082772107595063.m4a
Deleting original file ../data/audio/7520082772107595063.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


FAILED: 7519342755303230775
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7513759914842688814
[TikTok] 7513759914842688814: Downloading webpage
[info] 7513759914842688814: Downloading 1 format(s): bytevc1_1080p_226094-1
[download] Destination: ../data/audio/7513759914842688814.mp4
[download] 100% of    1.58MiB in 00:00:00 at 2.46MiB/s     
[ExtractAudio] Destination: ../data/audio/7513759914842688814.m4a
Deleting original file ../data/audio/7513759914842688814.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7513387289075174702
[TikTok] 7513387289075174702: Downloading webpage
[info] 7513387289075174702: Downloading 1 format(s): bytevc1_1080p_242404-1
[download] Destination: ../data/audio/7513387289075174702.mp4
[download] 100% of    1.62MiB in 00:00:00 at 1.78MiB/s     
[ExtractAudio] Destination: ../data/audio/7513387289075174702.m4a
Deleting original file ../data/audio/7513387289075174702.mp4 (pass -k to keep)
[TikTok] Ext

In [19]:
import os, json, shutil, subprocess, time, random
from pathlib import Path
import pandas as pd

# ========= CONFIG =========
profile_url = "https://www.tiktok.com/@humbletoker"
audio_dir = Path("../data/audio")

# If TikTok rate-limits or shows login walls, set this to your browser:
# e.g. "chrome", "firefox". If you don't need it, leave as None.
COOKIES_FROM_BROWSER = None  # "chrome"

# Politeness delay (helps avoid rate limits)
SLEEP_MIN, SLEEP_MAX = 0.8, 2.0

# ========= 0) Clear out current audio folder =========
audio_dir.mkdir(parents=True, exist_ok=True)
for p in audio_dir.glob("*"):
    if p.is_file():
        p.unlink()
print(f"Cleared audio folder: {audio_dir.resolve()}")

# ========= 1) Dump all video metadata from your profile =========
cmd = ["yt-dlp", "--dump-json", profile_url]
if COOKIES_FROM_BROWSER:
    cmd = ["yt-dlp", "--cookies-from-browser", COOKIES_FROM_BROWSER, "--dump-json", profile_url]

proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print("yt-dlp failed while listing profile videos.\n--- STDERR ---")
    print(proc.stderr[:4000])
    raise RuntimeError("Profile scrape failed (see stderr above).")

rows = []
for line in proc.stdout.splitlines():
    try:
        obj = json.loads(line)
    except json.JSONDecodeError:
        continue

    # Skip playlist wrapper objects (we want individual video entries)
    if obj.get("_type") == "playlist":
        continue

    video_id = obj.get("id")
    if not video_id:
        continue

    rows.append({
        "video_id": str(video_id),
        "video_url": obj.get("webpage_url") or obj.get("original_url"),
        "title": obj.get("title"),
        "upload_date": obj.get("upload_date"),
        "view_count": obj.get("view_count"),
        "like_count": obj.get("like_count"),
        "comment_count": obj.get("comment_count"),
        "repost_count": obj.get("repost_count"),
    })

videos = pd.DataFrame(rows).dropna(subset=["video_id", "video_url"]).drop_duplicates("video_id")
print(f"Found {len(videos)} videos (unique video_id).")

# Save for later use
out_csv = Path("../data/profile_videos_counts.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)
videos.to_csv(out_csv, index=False)
print(f"Saved counts to: {out_csv.resolve()}")

display_cols = ["video_id", "view_count", "like_count", "title", "video_url"]
print(videos[display_cols].sort_values("view_count", ascending=False).head(15).to_string(index=False))

# ========= 2) Download AUDIO for every video (saved as ../data/audio/<video_id>.m4a) =========
downloaded = skipped = failed = 0
fail_list = []

for _, r in videos.iterrows():
    vid = r["video_id"]
    url = r["video_url"]
    out_file = audio_dir / f"{vid}.m4a"

    if out_file.exists() and out_file.stat().st_size > 0:
        skipped += 1
        continue

    dl_cmd = [
        "yt-dlp",
        url,
        "--no-playlist",
        "-x",
        "--audio-format", "m4a",
        "--audio-quality", "0",
        "-o", str(audio_dir / f"{vid}.%(ext)s"),
        "--force-overwrites",
        "--no-warnings",
    ]
    if COOKIES_FROM_BROWSER:
        dl_cmd = [
            "yt-dlp",
            "--cookies-from-browser", COOKIES_FROM_BROWSER,
            url,
            "--no-playlist",
            "-x",
            "--audio-format", "m4a",
            "--audio-quality", "0",
            "-o", str(audio_dir / f"{vid}.%(ext)s"),
            "--force-overwrites",
            "--no-warnings",
        ]

    try:
        print(f"Downloading audio for {vid} ...")
        subprocess.run(dl_cmd, check=True)
        downloaded += 1
    except subprocess.CalledProcessError as e:
        failed += 1
        fail_list.append((vid, url, str(e)))
        print(f"FAILED {vid}")

    time.sleep(random.uniform(SLEEP_MIN, SLEEP_MAX))

print(f"\nAudio download complete. downloaded={downloaded}, skipped={skipped}, failed={failed}")
if fail_list:
    print("\nSample failures:")
    for vid, url, err in fail_list[:5]:
        print(f"- {vid} | {err}")
        print(f"  {url}")


Cleared audio folder: /Users/Logan/Desktop/TikTok Goodnight/data/audio
Found 52 videos (unique video_id).
Saved counts to: /Users/Logan/Desktop/TikTok Goodnight/data/profile_videos_counts.csv
           video_id  view_count  like_count                                                                    title                                                     video_url
7513079996349484319      379900        9940                                     Comment if you can’t speak English 👍 https://www.tiktok.com/@humbletoker/video/7513079996349484319
7520082772107595063      277000        9275 COMMENT IF YOU GOT A #GOODNIGHT !! Did anyone get it on super specific?? https://www.tiktok.com/@humbletoker/video/7520082772107595063
7516369352426474798      272800        8241                                     Comment to receive your goodnight ✌️ https://www.tiktok.com/@humbletoker/video/7516369352426474798
7517866483989417271      219600        5882                                        TikTok vi

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


FAILED 7542733065643232526
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7542350925122293006
[TikTok] 7542350925122293006: Downloading webpage
[info] 7542350925122293006: Downloading 1 format(s): bytevc1_1080p_295925-1
[download] Destination: ../data/audio/7542350925122293006.mp4
[download] 100% of    2.21MiB in 00:00:01 at 1.30MiB/s   
[ExtractAudio] Destination: ../data/audio/7542350925122293006.m4a
Deleting original file ../data/audio/7542350925122293006.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7541972554903309598
[TikTok] 7541972554903309598: Downloading webpage
[info] 7541972554903309598: Downloading 1 format(s): bytevc1_1080p_256854-1
[download] Destination: ../data/audio/7541972554903309598.mp4
[download] 100% of    1.91MiB in 00:00:01 at 1.23MiB/s     
[ExtractAudio] Destination: ../data/audio/7541972554903309598.m4a
Deleting original file ../data/audio/7541972554903309598.mp4 (pass -k to keep)
[TikTok] Extrac

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7535666289830890807
[TikTok] 7535666289830890807: Downloading webpage
[info] 7535666289830890807: Downloading 1 format(s): bytevc1_1080p_310809-1
[download] Destination: ../data/audio/7535666289830890807.mp4
[download] 100% of    2.30MiB in 00:00:01 at 1.32MiB/s   
[ExtractAudio] Destination: ../data/audio/7535666289830890807.m4a
Deleting original file ../data/audio/7535666289830890807.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7531214869471759647
[TikTok] 7531214869471759647: Downloading webpage
[info] 7531214869471759647: Downloading 1 format(s): bytevc1_1080p_247646-1
[download] Destination: ../data/audio/7531214869471759647.mp4
[download] 100% of    1.85MiB in 00:00:02 at 765.54KiB/s   
[ExtractAudio] Destination: ../data/audio/7531214869471759647.m4a
Deleting original file ../data/audio/7531214869471759647.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tikto

...

[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7513759914842688814
[TikTok] 7513759914842688814: Downloading webpage
[info] 7513759914842688814: Downloading 1 format(s): bytevc1_1080p_226094-1
[download] Destination: ../data/audio/7513759914842688814.mp4
[download] 100% of    1.58MiB in 00:00:00 at 1.68MiB/s   
[ExtractAudio] Destination: ../data/audio/7513759914842688814.m4a
Deleting original file ../data/audio/7513759914842688814.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7513425353528921387
[TikTok] 7513425353528921387: Downloading webpage
[info] 7513425353528921387: Downloading 1 format(s): bytevc1_1080p_220336-1
[download] Destination: ../data/audio/7513425353528921387.mp4
[download] 100% of    1.48MiB in 00:00:01 at 1.44MiB/s     
[ExtractAudio] Destination: ../data/audio/7513425353528921387.m4a
Deleting original file ../data/audio/7513425353528921387.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tikto

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7510421462252293406
[TikTok] 7510421462252293406: Downloading webpage
[info] 7510421462252293406: Downloading 1 format(s): h264_540p_404226-1
[download] Destination: ../data/audio/7510421462252293406.mp4
[download] 100% of    1.22MiB in 00:00:01 at 1.11MiB/s     
[ExtractAudio] Destination: ../data/audio/7510421462252293406.m4a
Deleting original file ../data/audio/7510421462252293406.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7510087154660314398
[TikTok] 7510087154660314398: Downloading webpage
[info] 7510087154660314398: Downloading 1 format(s): bytevc1_720p_177495-1
[download] Destination: ../data/audio/7510087154660314398.mp4
[download] 100% of    1.14MiB in 00:00:01 at 997.12KiB/s   
[ExtractAudio] Destination: ../data/audio/7510087154660314398.m4a
Deleting original file ../data/audio/7510087154660314398.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.c

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7508572618233105707
[TikTok] 7508572618233105707: Downloading webpage
[info] 7508572618233105707: Downloading 1 format(s): bytevc1_1080p_197982-1
[download] Destination: ../data/audio/7508572618233105707.mp4
[download] 100% of    1.30MiB in 00:00:01 at 1.28MiB/s   
[ExtractAudio] Destination: ../data/audio/7508572618233105707.m4a
Deleting original file ../data/audio/7508572618233105707.mp4 (pass -k to keep)
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7508222226462854446
[TikTok] 7508222226462854446: Downloading webpage
[info] 7508222226462854446: Downloading 1 format(s): bytevc1_720p_174292-1
[download] Destination: ../data/audio/7508222226462854446.mp4
[download] 100% of    1.26MiB in 00:00:00 at 1.55MiB/s   
[ExtractAudio] Destination: ../data/audio/7508222226462854446.m4a
Deleting original file ../data/audio/7508222226462854446.mp4 (pass -k to keep)

Audio download complete. downloaded=37, skip

In [20]:
import subprocess, time, random
from pathlib import Path

audio_dir = Path("../data/audio")
ffmpeg_loc = "/Users/Logan/miniforge3/bin"  # contains ffmpeg + ffprobe

# retry only videos missing .m4a
missing = []
for vid in videos["video_id"].astype(str):
    if not (audio_dir / f"{vid}.m4a").exists():
        missing.append(vid)

print("Missing audio files to retry:", len(missing))

downloaded = failed = 0
for vid in missing:
    url = f"https://www.tiktok.com/@humbletoker/video/{vid}"

    cmd = [
        "yt-dlp",
        url,
        "--no-playlist",
        "--ffmpeg-location", ffmpeg_loc,   # <-- key fix
        "-x",
        "--audio-format", "m4a",
        "--audio-quality", "0",
        "-o", str(audio_dir / f"{vid}.%(ext)s"),
        "--force-overwrites",
        "--no-warnings",
    ]

    try:
        print(f"Retrying {vid} ...")
        subprocess.run(cmd, check=True)
        downloaded += 1
    except subprocess.CalledProcessError as e:
        failed += 1
        print(f"STILL FAILED {vid}: {e}")

    time.sleep(random.uniform(0.8, 2.0))

print(f"\nRetry done. downloaded={downloaded}, failed={failed}")


Missing audio files to retry: 15
Retrying 7542733065643232526 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7542733065643232526
[TikTok] 7542733065643232526: Downloading webpage
[info] 7542733065643232526: Downloading 1 format(s): bytevc1_1080p_301834-1
Deleting existing file ../data/audio/7542733065643232526.mp4
[download] Destination: ../data/audio/7542733065643232526.mp4
[download] 100% of    2.28MiB in 00:00:02 at 1.00MiB/s   


ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


STILL FAILED 7542733065643232526: Command '['yt-dlp', 'https://www.tiktok.com/@humbletoker/video/7542733065643232526', '--no-playlist', '--ffmpeg-location', '/Users/Logan/miniforge3/bin', '-x', '--audio-format', 'm4a', '--audio-quality', '0', '-o', '../data/audio/7542733065643232526.%(ext)s', '--force-overwrites', '--no-warnings']' returned non-zero exit status 1.
Retrying 7536764777184496951 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7536764777184496951
[TikTok] 7536764777184496951: Downloading webpage
[info] 7536764777184496951: Downloading 1 format(s): bytevc1_1080p_303245-1
Deleting existing file ../data/audio/7536764777184496951.mp4
[download] Destination: ../data/audio/7536764777184496951.mp4
[download] 100% of    2.23MiB in 00:00:01 at 1.57MiB/s   
[ExtractAudio] Destination: ../data/audio/7536764777184496951.m4a
Deleting original file ../data/audio/7536764777184496951.mp4 (pass -k to keep)
Retrying 7529733595515129102 ...
[TikTok] Extracting URL: htt

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


STILL FAILED 7529733595515129102: Command '['yt-dlp', 'https://www.tiktok.com/@humbletoker/video/7529733595515129102', '--no-playlist', '--ffmpeg-location', '/Users/Logan/miniforge3/bin', '-x', '--audio-format', 'm4a', '--audio-quality', '0', '-o', '../data/audio/7529733595515129102.%(ext)s', '--force-overwrites', '--no-warnings']' returned non-zero exit status 1.
Retrying 7528949556499844365 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7528949556499844365
[TikTok] 7528949556499844365: Downloading webpage
[info] 7528949556499844365: Downloading 1 format(s): bytevc1_1080p_307647-1
Deleting existing file ../data/audio/7528949556499844365.mp4
[download] Destination: ../data/audio/7528949556499844365.mp4
[download] 100% of    2.01MiB in 00:00:01 at 1.79MiB/s   
STILL FAILED 7528949556499844365: Command '['yt-dlp', 'https://www.tiktok.com/@humbletoker/video/7528949556499844365', '--no-playlist', '--ffmpeg-location', '/Users/Logan/miniforge3/bin', '-x', '--audio-for

...

Retrying 7526766564419341623 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7526766564419341623
[TikTok] 7526766564419341623: Downloading webpage
[info] 7526766564419341623: Downloading 1 format(s): bytevc1_1080p_347991-1
Deleting existing file ../data/audio/7526766564419341623.mp4
[download] Destination: ../data/audio/7526766564419341623.mp4
[download] 100% of    2.58MiB in 00:00:03 at 689.40KiB/s   
[ExtractAudio] Destination: ../data/audio/7526766564419341623.m4a
Deleting original file ../data/audio/7526766564419341623.mp4 (pass -k to keep)
Retrying 7525640287578787103 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7525640287578787103
[TikTok] 7525640287578787103: Downloading webpage
[info] 7525640287578787103: Downloading 1 format(s): bytevc1_1080p_508482-1
Deleting existing file ../data/audio/7525640287578787103.mp4
[download] Destination: ../data/audio/7525640287578787103.mp4
[download] 100% of    3.15MiB in 00:00:01 at 1.77MiB/s   

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


Retrying 7520082772107595063 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7520082772107595063
[TikTok] 7520082772107595063: Downloading webpage
[info] 7520082772107595063: Downloading 1 format(s): bytevc1_1080p_266677-1
Deleting existing file ../data/audio/7520082772107595063.mp4
[download] Destination: ../data/audio/7520082772107595063.mp4
[download] 100% of    2.42MiB in 00:00:02 at 978.42KiB/s   
[ExtractAudio] Destination: ../data/audio/7520082772107595063.m4a
Deleting original file ../data/audio/7520082772107595063.mp4 (pass -k to keep)
Retrying 7518596851226922295 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7518596851226922295
[TikTok] 7518596851226922295: Downloading webpage
[info] 7518596851226922295: Downloading 1 format(s): bytevc1_1080p_238132-1
Deleting existing file ../data/audio/7518596851226922295.mp4
[download] Destination: ../data/audio/7518596851226922295.mp4
[download] 100% of    1.82MiB in 00:00:01 at 991.97KiB/s 

ERROR: Postprocessing: WARNING: unable to obtain file audio codec with ffprobe


Retrying 7517115051220700471 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7517115051220700471
[TikTok] 7517115051220700471: Downloading webpage
[info] 7517115051220700471: Downloading 1 format(s): bytevc1_1080p_285674-1
Deleting existing file ../data/audio/7517115051220700471.mp4
[download] Destination: ../data/audio/7517115051220700471.mp4
[download] 100% of    2.10MiB in 00:00:01 at 1.52MiB/s     
[ExtractAudio] Destination: ../data/audio/7517115051220700471.m4a
Deleting original file ../data/audio/7517115051220700471.mp4 (pass -k to keep)
Retrying 7514160603687128366 ...
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7514160603687128366
[TikTok] 7514160603687128366: Downloading webpage
[info] 7514160603687128366: Downloading 1 format(s): bytevc1_1080p_202914-1
Deleting existing file ../data/audio/7514160603687128366.mp4
[download] Destination: ../data/audio/7514160603687128366.mp4
[download] 100% of    1.56MiB in 00:00:01 at 1.47MiB/s   

In [21]:
import subprocess
from pathlib import Path

audio_dir = Path("../data/audio")
ffmpeg = "/Users/Logan/miniforge3/bin/ffmpeg"

# find what's still missing
missing = [vid for vid in videos["video_id"].astype(str) if not (audio_dir / f"{vid}.m4a").exists()]
print("Still missing:", len(missing))
print(missing)

recovered = 0
still_failed = []

for vid in missing:
    url = f"https://www.tiktok.com/@humbletoker/video/{vid}"
    mp4_path = audio_dir / f"{vid}.mp4"
    m4a_path = audio_dir / f"{vid}.m4a"

    try:
        # 1) download mp4 only (no extraction)
        subprocess.run([
            "yt-dlp", url,
            "--no-playlist",
            "-f", "mp4",
            "-o", str(mp4_path),
            "--force-overwrites",
            "--no-warnings"
        ], check=True)

        # 2) convert mp4 -> m4a via ffmpeg (robust)
        subprocess.run([
            ffmpeg, "-y",
            "-i", str(mp4_path),
            "-vn",
            "-ac", "1",
            "-ar", "16000",
            str(m4a_path)
        ], check=True)

        # 3) clean up mp4
        if mp4_path.exists():
            mp4_path.unlink()

        recovered += 1
        print("Recovered:", vid)

    except subprocess.CalledProcessError as e:
        still_failed.append(vid)
        print("Still failed:", vid, e)

print(f"\nRecovered {recovered}/{len(missing)}. Still failed: {len(still_failed)}")
if still_failed:
    print("Still failing IDs:", still_failed)


Still missing: 6
['7542733065643232526', '7529733595515129102', '7528949556499844365', '7527108845336792375', '7525640287578787103', '7518237738047458573']
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7542733065643232526
[TikTok] 7542733065643232526: Downloading webpage
[info] 7542733065643232526: Downloading 1 format(s): bytevc1_1080p_357046-1
Deleting existing file ../data/audio/7542733065643232526.mp4
[download] Destination: ../data/audio/7542733065643232526.mp4
[download] 100% of    2.67MiB in 00:00:01 at 1.35MiB/s     


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 19.1.7
  configuration: --prefix=/Users/Logan/miniforge3 --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1766458828561/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libvorbi

Recovered: 7542733065643232526
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7529733595515129102
[TikTok] 7529733595515129102: Downloading webpage
[info] 7529733595515129102: Downloading 1 format(s): bytevc1_1080p_258885-1
Deleting existing file ../data/audio/7529733595515129102.mp4
[download] Destination: ../data/audio/7529733595515129102.mp4
[download] 100% of    1.91MiB in 00:00:01 at 1.36MiB/s   


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 19.1.7
  configuration: --prefix=/Users/Logan/miniforge3 --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1766458828561/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libvorbi

Recovered: 7529733595515129102
[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7528949556499844365
[TikTok] 7528949556499844365: Downloading webpage
[info] 7528949556499844365: Downloading 1 format(s): bytevc1_1080p_327434-1
Deleting existing file ../data/audio/7528949556499844365.mp4
[download] Destination: ../data/audio/7528949556499844365.mp4
[download] 100% of    2.39MiB in 00:00:01 at 1.25MiB/s     


...

ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 19.1.7
  configuration: --prefix=/Users/Logan/miniforge3 --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1766458828561/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libvorbi

[TikTok] Extracting URL: https://www.tiktok.com/@humbletoker/video/7518237738047458573
[TikTok] 7518237738047458573: Downloading webpage
[info] 7518237738047458573: Downloading 1 format(s): bytevc1_1080p_291001-1
Deleting existing file ../data/audio/7518237738047458573.mp4
[download] Destination: ../data/audio/7518237738047458573.mp4
[download] 100% of    2.50MiB in 00:00:02 at 1.23MiB/s     


ffmpeg version 8.0.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 19.1.7
  configuration: --prefix=/Users/Logan/miniforge3 --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1766458828561/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libvorbi

Recovered: 7518237738047458573

Recovered 5/6. Still failed: 1
Still failing IDs: ['7525640287578787103']


[out#0/ipod @ 0x1256042c0] video:0KiB audio:640KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.819908%
size=     645KiB time=00:01:12.02 bitrate=  73.4kbits/s speed= 252x elapsed=0:00:00.28    
[aac @ 0x125605460] Qavg: 34269.742
